In [17]:
# IMPORTS
# ==========================================

# Libraries for system handling and the reinforcement learning environment
import os

# Libraries for mathematical operations and randomness
import numpy as np
import random
import logging  # For message logging and debugging
import itertools  # For generating combinations and permutations
import statistics
import cv2
import csv
import math
import heapq
import json


# Library for DataFrame operations
import pandas as pd

# Library for deep learning operations with TensorFlow
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, Flatten, Dense, InputLayer, MultiHeadAttention, LayerNormalization, Reshape, MultiHeadAttention, GlobalAveragePooling2D, TimeDistributed, MaxPooling2D, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam  # Optimizer for adjusting neural network weights
from tensorflow.keras.losses import MeanSquaredError, Huber
from tensorflow.keras.regularizers import l2  # L2 regularization for controlling overfitting
from tensorflow.keras.models import load_model, Model  # Load pretrained models, 



from tensorflow.keras.layers import (InputLayer, Conv2D, BatchNormalization, Flatten, 
                                     Dense, Dropout, Activation, Concatenate, Multiply, Layer)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanSquaredError, Huber
import tensorflow.keras.backend as K


# Check available physical devices (for example, a GPU)
#device_lib.list_local_devices()
tf.config.list_physical_devices('GPU')



[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

### CLASS
----------

In [18]:
#Class: Agents
# ==========================================

class Agent:
    def __init__(self, x, y, env, breed, channel):
        self.x = x
        self.y = y
        self.env = env
        self.breed = breed
        self.channel = channel
        self.is_alive = True
        self.is_done = False
        self.current_target = None
        self.current_ally = None
        self.last_coord = self.x, self.y
        self.kid = False

        # Category         | Breed | Color  |     Channel      | Color - Att | Channel - Att
        # ---------------------------------------------------------------------------
        # Predator         |   0   | Red    |    [1, 0, 0]     |   Yellow     | [1, 1, 0]
        # Vegetation       |   1   | Green  |    [0, 1, 0]     |  --------    | ---------
        # Prey             |   2   | Blue   |    [0, 0, 1]     |   Cyan       | [0, 1, 1]
        # Outside Grid     |   -   | White  |    [1, 1, 1]     |  --------    | ---------
        # Empty Grid Cell  |   -   | Black  |    [0, 0, 0]     |  --------    | ---------
        # Reserved Training|   -   | Gray   | [0.5, 0.5, 0.5]  |  --------    | ---------

    def reset(self, env):
        self.x, self.y = env.new_position()
        self.is_alive = True
        self.target = None
        self.is_done = False
        self.current_target = None
        self.current_ally = None
        
        # Restore `channel` to the default color based on `breed`
        if self.breed == 0:  # Predator
            self.channel = [1.0, 0.0, 0.0]
        elif self.breed == 2:  # Prey
            self.channel = [0.0, 0.0, 1.0]

    def step(self, action):
        penalty = self.env.move_agent(self, action)
        done, reward, ler = self.check_goal()
        state = self.env.render_agent(self)

        return state, reward + penalty, done, ler
    
class Prey(Agent):
    def __init__(self, x, y, env, id, nn = None, is_on = True):
        super().__init__(x, y, env, breed=2, channel=[0.0, 0.0, 1.0])
        self.name = f"Prey_{id}"
        self.in_danger = False
        self.is_on = is_on
        self.model_nn = nn

    def check_goal(self):
        done = False
        reward = 5.0  # Default reward to encourage safe exploration.
        feedback = ""
        target_detected = False
        ally_detected = False

        closest_target_distance = float('inf')
        closest_ally_distance = float('inf')

        # Find the nearest living agent, prey, or predator, or keep focus on the current target.
        for agent in self.env.agents:
            # Check whether it is a predator
            if agent.breed == 0 and agent.is_alive:
                distance = self.env.chebyshev_distance(self.x, self.y, agent.x, agent.y)
                if distance <= self.env.ray and (self.current_target is None or distance < closest_target_distance):
                    closest_target_distance = distance
                    self.current_target = agent
                    target_detected = True

            # Check whether it is a prey agent with predator information
            elif agent.breed == 2 and agent.is_alive and agent.channel == [0, 1, 1]:
                distance = self.env.chebyshev_distance(self.x, self.y, agent.x, agent.y)
                if distance <= self.env.ray and (self.current_ally is None or distance < closest_ally_distance):
                    closest_ally_distance = distance
                    self.current_ally = agent
                    ally_detected = True

        
        # Check whether the prey escaped from the predator
        if self.in_danger and closest_target_distance > 3:
            done = True
            self.in_danger = False
            reward += 20.0  # Maximum reward for escaping danger
            feedback = f"[PREY]: Evasao do alvo"
        
        # Reward and penalty criteria
        elif target_detected and 0 < closest_target_distance <= 3:
            # Penalty for proximity to a predator
            self.in_danger = True
            if closest_target_distance == 3:
                reward -= 1.0
            elif closest_target_distance == 2:
                reward -= 3.0
            elif closest_target_distance == 1:
                reward -= 5.0
            feedback = f"[PREY]: Proximidade predador: {closest_target_distance}"

        elif ally_detected and not target_detected and 0 < closest_ally_distance <= 3:
            # Reward for proximity to an ally (when no predator is in the field of view)
            if closest_ally_distance == 3:
                reward -= 0.1
            elif closest_ally_distance == 2:
                reward -= 0.3
            elif closest_ally_distance == 1:
                reward -= 0.5
            feedback += f"[PREY]: Proximidade aliado: {closest_ally_distance}"

        else:
            feedback += "[PREY]: Explorando mapa"


        # Update `channel`
        for agent in self.env.agents:
            if agent.is_alive and agent.breed == 0:
                distance = self.env.chebyshev_distance(self.x, self.y, agent.x, agent.y)
                if distance <= self.env.ray:
                    self.channel = [0.0, 1.0, 1.0]
                    break
                else:
                    self.channel = [0.0, 0.0, 1.0]

        return done, reward, feedback

class Predator(Agent):
    def __init__(self, x, y, env, id, nn = None, is_on = True):
        super().__init__(x, y, env, breed=0, channel=[1.0, 0.0, 0.0])
        self.name = f"Predador_{id}"
        self.last_hunt = None
        self.is_on = is_on

    def check_goal(self):
        done = False
        reward = -0.05  # Small penalty to encourage movement
        feedback = ""
        target_detected = False
        ally_detected = False

        closest_target_distance = float('inf')
        closest_ally_distance = float('inf')

        # Find the nearest living agent, prey, or predator, or keep focus on the current target.
        for agent in self.env.agents:
            # Check whether it is a prey agent
            if agent.is_alive and agent.breed == 2:
                distance = self.env.chebyshev_distance(self.x, self.y, agent.x, agent.y)
                if distance <= self.env.ray and (self.current_target is None or distance < closest_target_distance):
                    closest_target_distance = distance
                    self.current_target = agent
                    target_detected = True

            # Check whether it is a predator with prey information
            elif agent.breed == 0 and agent.is_alive and agent.channel == [1.0, 1.0, 0.0]:
                distance = self.env.chebyshev_distance(self.x, self.y, agent.x, agent.y)
                if distance <= self.env.ray and (self.current_ally is None or distance < closest_ally_distance):
                    closest_ally_distance = distance
                    self.current_ally = agent
                    ally_detected = True

        # Reward criteria
        if target_detected and closest_target_distance == 0:
            done = True
            reward += 20.0  # Maximum reward for capturing the prey
            self.current_target.is_alive = False
            feedback = f"[PREDATOR]: Alvo capturado"
            # Removal from the list is handled by the training class
            self.current_target = None

        elif target_detected and 0 < closest_target_distance <= 3:
            # Reward for proximity to the prey
            if closest_target_distance == 3:
                reward += 0.5
            elif closest_target_distance == 2:
                reward += 1.0
            elif closest_target_distance == 1:
                reward += 2.0
            feedback = f"[PREDATOR]: Proximidade presa: {closest_target_distance}"

        elif ally_detected and 0 < closest_ally_distance <= 3:
            # Reward for proximity to an ally that has prey information
            if closest_ally_distance == 3:
                reward += 0.1
            elif closest_ally_distance == 2:
                reward += 0.3
            elif closest_ally_distance == 1:
                reward += 0.5
            feedback = f"[PREDATOR]: Proximidade aliado: {closest_ally_distance}"

        else:
            feedback += "[PREDATOR]: Explorando mapa"

        # Update `channel`
        for agent in self.env.agents:
            if agent.is_alive and agent.breed == 2:
                distance = self.env.chebyshev_distance(self.x, self.y, agent.x, agent.y)
                if distance <= self.env.ray:
                    self.channel = [1.0, 1.0, 0.0]
                    break
                else:
                    self.channel = [1.0, 0.0, 0.0]

        return done, reward, feedback

In [19]:
#Class: Obstacle
# ==========================================

class Obstacle:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.channel = [0.0, 1.0, 0.0]  # Green color for the obstacle (RGB)

    def reset(self, env):
        self.channel = [0.0, 1.0, 0.0]

        # Choose a random free position in the grid
        position = env.new_position()
        if position:
            self.x, self.y = position
        else:
            raise ValueError("Não foi possível encontrar uma posição livre para o obstáculo.")

In [20]:
#Class: Environment
# ==========================================

class Env:
    def __init__(self, sizeX, sizeY, ray=3):
        self.sizeX = sizeX
        self.sizeY = sizeY
        self.ray = ray
        self.agents = []
        self.objects = []
        self.obstacles = []  # Separate list for obstacles
        self.actions = 9  # Number of possible actions


    def new_position(self):
        # Create a list of all possible positions.
        iterables = [range(self.sizeX), range(self.sizeY)]
        points = list(itertools.product(*iterables))

        # Create a list of positions currently occupied by agents and obstacles.
        current_positions = [(agent.x, agent.y) for agent in self.agents]
        current_positions += [(obstacle.x, obstacle.y) for obstacle in self.obstacles]

        # Filter possible positions by removing occupied ones.
        available_points = [point for point in points if point not in current_positions]

        # Randomly choose one of the available positions.
        if available_points:
            return random.choice(available_points)
        else:
            # Log a message indicating that no positions are available.
            logging.info("Não foi possível encontrar uma nova posição disponível.")
            return None

    def add_obstacle(self, obstacle):
        # Assign an available position to the obstacle
        position = self.new_position()
        if position:
            obstacle.x, obstacle.y = position  # Assign coordinates to the obstacle
            self.objects.append(obstacle)
        else:
            logging.info("Não foi possível adicionar um novo obstáculo: nenhuma posição disponível.")

    def add_agent(self, agent):
        # Assign an available position to the agent
        position = self.new_position()
        if position:
            agent.x, agent.y = position  # Assign coordinates to the agent
            self.objects.append(agent)
        else:
            logging.info("Não foi possível adicionar um novo agente: nenhuma posição disponível.")

    def reset(self):
        # Check whether the object list is empty
        if not self.objects:
            logging.info("Não é possível prosseguir: nenhum objeto foi adicionado ao ambiente.")
            return  # Stop the method if `self.objects` is empty

        # Clear the agent and obstacle lists to reset the environment
        self.agents = []
        self.obstacles = []

        # Shuffle the object list to vary the order of elements in the environment
        random.shuffle(self.objects)

        # Add agents and obstacles to the appropriate list and reset them
        for item in self.objects:
            item.reset(self)  # Reposition each object in the environment
            if isinstance(item, Agent):  # Assuming an `Agent` class
                self.agents.append(item)
            elif isinstance(item, Obstacle):  # Assuming an `Obstacle` class
                self.obstacles.append(item)

    def is_position_empty_and_valid(self, x, y):
        # Check whether the position is within the environment boundaries
        if x < 0 or x >= self.sizeX or y < 0 or y >= self.sizeY:
            return False  # The position is outside the environment boundaries

        # Check whether the position is occupied by an agent
        for agent in self.agents:
            if agent.x == x and agent.y == y:
                return False  # The position is occupied

        # Check whether the position is occupied by an obstacle
        for obstacle in self.obstacles:
            if obstacle.x == x and obstacle.y == y:
                return False  # The position is occupied by an obstacle

    def get_agent_at_position(self, x, y):
        for agent in self.agents:
            if agent.x == x and agent.y == y:
                return agent
        return None

    def move_agent(self, agent, action):
        # Initialize the default penalty and penalty values
        ZERO = 0.0
        PENALIZE = -10.0
        direction = action

        # Initialize movement increments
        new_x, new_y = 0, 0

        # Set movement increments based on direction
        if direction == 0:  # Up
            new_y = -1
        elif direction == 1:  # Up and right (diagonal)
            new_x = 1
            new_y = -1
        elif direction == 2:  # Right
            new_x = 1
        elif direction == 3:  # Down and right (diagonal)
            new_x = 1
            new_y = 1
        elif direction == 4:  # Down
            new_y = 1
        elif direction == 5:  # Down and left (diagonal)
            new_x = -1
            new_y = 1
        elif direction == 6:  # Left
            new_x = -1
        elif direction == 7:  # Up and left (diagonal)
            new_x = -1
            new_y = -1
        elif direction == 8:  # Remain stationary
            new_x = 0
            new_y = 0

        # Calculate the agent's new absolute position
        target_x = agent.x + new_x
        target_y = agent.y + new_y

        # Check whether the new position contains an obstacle
        if any(obstacle.x == target_x and obstacle.y == target_y for obstacle in self.obstacles):
            print("Agente tentou ocupar um obstáculo!")
            return PENALIZE  # Penalty for attempting to occupy an obstacle position

        # Check whether the movement is within the environment boundaries
        if target_x < 0 or target_x >= self.sizeX or target_y < 0 or target_y >= self.sizeY:
            print("Agente fora do limite!")
            return PENALIZE

        # Check whether the new position contains another agent
        other_agent = self.get_agent_at_position(target_x, target_y)
        if other_agent:
            # Logic for prey agents
            if agent.breed == 2:  # Prey
                print("Presa não pode ocupar a posição de outro agente!")
                return PENALIZE  # Penalty for attempting to occupy another agent's position

            # Logic for predator agents
            elif agent.breed == 0:  # Predator
                if other_agent.breed == 0:  # Another predator is in the position
                    print("Predador não pode ocupar a posição de outro predador!")
                    return PENALIZE  # Penalty for attempting to occupy another predator's position

        # Update the agent's position if the movement is valid
        agent.x, agent.y = target_x, target_y
        return ZERO

    def render_env(self):
        a = np.zeros([self.sizeY, self.sizeX, 3])

        for agent in self.agents:
            if agent.x is not None and agent.y is not None:
                a[agent.y, agent.x, :] = agent.channel

        for obstacle in self.obstacles:
            a[obstacle.y, obstacle.x, :] = obstacle.channel  # Render obstacles

        return a

    def render_agent(self, agent):
        # Render the environment to obtain the current RGB matrix
        a = self.render_env()

        # Calculate the crop size based on `self.ray`
        recorte_tamanho = 2 * self.ray + 1

        # Initialize the temporary crop with yellow
        recorte_temp = np.ones((recorte_tamanho, recorte_tamanho, 3)) * np.array([1.0, 1.0, 1.0])  # Outside the grid - white

        # Calculate crop coordinates within the environment
        inicio_x = agent.x - self.ray
        inicio_y = agent.y - self.ray
        fim_x = inicio_x + recorte_tamanho
        fim_y = inicio_y + recorte_tamanho

        # Calculate overlap boundaries between the crop and the environment
        sobreposicao_inicio_x = max(inicio_x, 0)
        sobreposicao_inicio_y = max(inicio_y, 0)
        sobreposicao_fim_x = min(fim_x, self.sizeX)
        sobreposicao_fim_y = min(fim_y, self.sizeY)

        # Calculate destination indices in the temporary crop
        destino_inicio_x = sobreposicao_inicio_x - inicio_x
        destino_inicio_y = sobreposicao_inicio_y - inicio_y
        destino_fim_x = destino_inicio_x + sobreposicao_fim_x - sobreposicao_inicio_x
        destino_fim_y = destino_inicio_y + sobreposicao_fim_y - sobreposicao_inicio_y

        # Copy the environment overlap into the temporary crop
        recorte_temp[destino_inicio_y:destino_fim_y, destino_inicio_x:destino_fim_x] = \
            a[sobreposicao_inicio_y:sobreposicao_fim_y, sobreposicao_inicio_x:sobreposicao_fim_x]

        # Paint the central crop element white, adjusting its position based on `self.ray`
        centro = self.ray
        recorte_temp[centro, centro, :] = np.array([0.5, 0.5, 0.5])  # Gray - highlight the agent being trained

        return recorte_temp

    def population_count(self):
        """Retorna a quantidade de presas, predadores e obstáculos no ambiente."""
        predator_count = 0
        prey_count = 0

        for agent in self.agents:
            if agent.is_alive is True:
                if agent.breed == 0:
                    predator_count += 1
                elif agent.breed == 2:
                    prey_count += 1

        obstacle_count = len(self.obstacles)  # Count all obstacles in `self.obstacles`
        return prey_count, predator_count, obstacle_count

    def remove_agent(self, agent):
        self.agents.remove(agent)

    @staticmethod
    def chebyshev_distance(x1, y1, x2, y2):
        distance = max(abs(x2 - x1), abs(y2 - y1))
        return distance


### MODELS
----------------------------------

In [21]:
# Support
# ==========================================

lr = 0.0001
l2_regularization = 0.01

In [22]:
# Class - RADAR
# ==========================================

class ColorCombDepthwiseConv2D(Layer):
    def __init__(self, kernel_size=(7, 7), activation='relu', padding='same', **kwargs):
        super(ColorCombDepthwiseConv2D, self).__init__(**kwargs)
        self.kernel_size = kernel_size
        self.activation = activation
        self.padding = padding

        # Individual convolutions for pure colors
        self.conv_r = Conv2D(1, kernel_size=self.kernel_size, padding=self.padding, activation=None, name="conv_r")
        self.bn_r = BatchNormalization(name="bn_r")
        
        self.conv_g = Conv2D(1, kernel_size=self.kernel_size, padding=self.padding, activation=None, name="conv_g")
        self.bn_g = BatchNormalization(name="bn_g")
        
        self.conv_b = Conv2D(1, kernel_size=self.kernel_size, padding=self.padding, activation=None, name="conv_b")
        self.bn_b = BatchNormalization(name="bn_b")

        # Convolutions for specific combinations (magenta, cyan, and yellow)
        self.conv_magenta = Conv2D(1, kernel_size=self.kernel_size, padding=self.padding, activation=None, name="conv_magenta")
        self.bn_magenta = BatchNormalization(name="bn_magenta")
        
        self.conv_cyan = Conv2D(1, kernel_size=self.kernel_size, padding=self.padding, activation=None, name="conv_cyan")
        self.bn_cyan = BatchNormalization(name="bn_cyan")
        
        self.conv_yellow = Conv2D(1, kernel_size=self.kernel_size, padding=self.padding, activation=None, name="conv_yellow")
        self.bn_yellow = BatchNormalization(name="bn_yellow")

    def call(self, inputs, training=False):
        r, g, b = tf.split(inputs, num_or_size_splits=3, axis=-1)

        # Individual convolutions with batch normalization
        r_out = self.bn_r(self.conv_r(r), training=training)
        g_out = self.bn_g(self.conv_g(g), training=training)
        b_out = self.bn_b(self.conv_b(b), training=training)

        # Channel combinations with batch normalization
        magenta_out = self.bn_magenta(self.conv_magenta(r + b), training=training)
        cyan_out = self.bn_cyan(self.conv_cyan(g + b), training=training)
        yellow_out = self.bn_yellow(self.conv_yellow(r + g), training=training)

        # Concatenate outputs
        outputs = Concatenate(axis=-1, name="concat_colors")([r_out, g_out, b_out, magenta_out, cyan_out, yellow_out])

        # Activation
        outputs = Activation(self.activation, name="activation_colors")(outputs)
        return outputs


class SpatialAttentionModule(Layer):
    def __init__(self, kernel_size=4):
        super(SpatialAttentionModule, self).__init__()
        self.conv = Conv2D(1, kernel_size=kernel_size, padding='same', activation=None, name="attention_conv")
    
    def call(self, inputs):
        avg_pool = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        squared_inputs = K.square(inputs)
        l2_pool = tf.sqrt(tf.reduce_mean(squared_inputs, axis=-1, keepdims=True) + 1e-6)

        concat = Concatenate(axis=-1, name="concat_attention")([avg_pool, l2_pool])
        attention_map = self.conv(concat)
        attention_map = Activation('sigmoid', name="sigmoid_attention")(attention_map)
        return Multiply(name="apply_attention")([inputs, attention_map])



In [23]:
# Model: nn_dqn
# ==========================================

class nn_dqn(tf.keras.Model):
    def __init__(self, num_actions=9, input_shape=(7, 7, 3)):
        super(nn_dqn, self).__init__()
        self.num_actions = num_actions
        self.optimizer = Adam(learning_rate=lr)
        self.loss_fn = MeanSquaredError()

        # Input and convolutional layers
        self.input_layer = InputLayer(input_shape=input_shape)

        # Convolutional layers
        self.conv1 = Conv2D(32, (4, 4), strides=(1, 1), activation=None, padding='same', name="conv1_layer")
        self.bn1 = BatchNormalization(name="bn1_layer")
        self.dropout_conv1 = Dropout(0.3, name="dropout_conv1_layer")  # Dropout after conv1

        self.conv2 = Conv2D(64, (3, 3), strides=(1, 1), activation=None, padding='same', name="conv2_layer")
        self.bn2 = BatchNormalization(name="bn2_layer")
        self.dropout_conv2 = Dropout(0.3, name="dropout_conv2_layer")  # Dropout after conv2

        # Flattening and dense layers
        self.flatten = Flatten(name="flatten_layer")
        self.dropout_flatten = Dropout(0.4, name="dropout_flatten_layer")  # Dropout after flattening

        self.dense1 = Dense(64, activation='relu', name="dense1_layer", kernel_regularizer=l2(l2_regularization))
        self.dropout_dense1 = Dropout(0.5, name="dropout_dense1_layer")  # Dropout after dense1
        
        self.dense2 = Dense(32, activation='relu', name="dense2_layer", kernel_regularizer=l2(l2_regularization))
        self.dropout_dense2 = Dropout(0.5, name="dropout_dense2_layer")  # Dropout after dense2

        self.dense_output = Dense(num_actions, activation='linear', name="dense_output_layer", kernel_regularizer=l2(l2_regularization))

    def call(self, inputs, training=False):
        x = self.input_layer(inputs)

        # Convolutional layer 1
        x = self.conv1(x)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x, name="relu1")
        x = self.dropout_conv1(x, training=training)  # Apply dropout after conv1

        # Convolutional layer 2
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x, name="relu2")
        x = self.dropout_conv2(x, training=training)  # Apply dropout after conv2

        # Spatial attention and flattening
        x = self.flatten(x)
        x = self.dropout_flatten(x, training=training)  # Apply dropout after flattening

        # Dense layers
        x = self.dense1(x)
        x = self.dropout_dense1(x, training=training)  # Apply dropout after dense1
        x = self.dense2(x)
        x = self.dropout_dense2(x, training=training)  # Apply dropout after dense2

        Q_values = self.dense_output(x)
        return Q_values


    def training_step(self, batch_data):
        states, actions, targetQ = batch_data
        with tf.GradientTape() as tape:
            Q_values = self(states, training=True)
            actions_onehot = tf.one_hot(actions, self.num_actions, dtype=tf.float32)
            Q = tf.reduce_sum(Q_values * actions_onehot, axis=1)
            loss = self.loss_fn(targetQ, Q)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return loss

    def predict_action(self, state):
        Q_values = self(state)
        return tf.argmax(Q_values, axis=1)[0].numpy()  # Return the action with the highest Q-value as a Python number

    def save_model(self, file_path):
        self.save(file_path)
        print(f"Modelo salvo em: {file_path}")

    def load_model(self, file_path):
        model_loaded = tf.keras.models.load_model(file_path)
        return model_loaded
    


In [24]:
# Model: nn_per
# ==========================================

class nn_per(tf.keras.Model):
    def __init__(self, num_actions=9, input_shape=(7, 7, 3), discount_factor=0.99):
        super(nn_per, self).__init__()
        self.num_actions = num_actions
        self.discount_factor = discount_factor
        self.optimizer = Adam(learning_rate=lr)
        self.loss_fn = MeanSquaredError()

        # Input and color-combination layer
        self.input_layer = InputLayer(input_shape=input_shape, name="input_layer")

        # Convolutional layers
        self.conv1 = Conv2D(32, (4, 4), strides=(1, 1), activation=None, padding='same', name="conv1_layer")
        self.bn1 = BatchNormalization(name="bn1_layer")
        self.dropout_conv1 = Dropout(0.3, name="dropout_conv1_layer")

        self.conv2 = Conv2D(64, (3, 3), strides=(1, 1), activation=None, padding='same', name="conv2_layer")
        self.bn2 = BatchNormalization(name="bn2_layer")
        self.dropout_conv2 = Dropout(0.3, name="dropout_conv2_layer")

        # Flattening and dense layers
        self.flatten = Flatten(name="flatten_layer")
        self.dropout_flatten = Dropout(0.3, name="dropout_flatten_layer")

        self.dense1 = Dense(64, activation='relu', name="dense1_layer", kernel_regularizer=l2(l2_regularization))
        self.dropout_dense1 = Dropout(0.4, name="dropout_dense1_layer")

        self.dense2 = Dense(32, activation='relu', name="dense2_layer", kernel_regularizer=l2(l2_regularization))
        self.dropout_dense2 = Dropout(0.4, name="dropout_dense2_layer")

        self.dense_output = Dense(num_actions, activation='linear', name="dense_output_layer", kernel_regularizer=l2(l2_regularization))

    def call(self, inputs, training=False):
        x = self.input_layer(inputs)

        x = self.conv1(x)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x, name="relu1")
        x = self.dropout_conv1(x, training=training)  # Apply dropout after conv1

        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x, name="relu2")
        x = self.dropout_conv2(x, training=training)  # Apply dropout after conv2

        x = self.flatten(x)
        x = self.dropout_flatten(x, training=training)  # Apply dropout after flattening

        x = self.dense1(x)
        x = self.dropout_dense1(x, training=training)  # Apply dropout after dense1
        x = self.dense2(x)
        x = self.dropout_dense2(x, training=training)  # Apply dropout after dense2

        Q_values = self.dense_output(x)
        return Q_values
    

    def training_step(self, batch_data):
        states, actions, targetQ, weights = batch_data
        with tf.GradientTape() as tape:
            Q_values = self(states, training=True)
            actions_onehot = tf.one_hot(actions, self.num_actions, dtype=tf.float32)
            Q = tf.reduce_sum(Q_values * actions_onehot, axis=1)
            td_errors = targetQ - Q
            weighted_loss = tf.reduce_mean(weights * tf.square(td_errors))

        grads = tape.gradient(weighted_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        return weighted_loss  # Loss only


    def predict_action(self, state):
        Q_values = self(state)
        return tf.argmax(Q_values, axis=1)[0].numpy()  # Return the action with the highest Q-value as a Python number

    def save_model(self, file_path):
        self.save(file_path)
        print(f"Modelo salvo em: {file_path}")

    def load_model(self, file_path):
        model_loaded = tf.keras.models.load_model(file_path)
        return model_loaded




In [25]:
# Model: nn_dueling
# ==========================================

class nn_dueling(tf.keras.Model):
    def __init__(self, num_actions=9, input_shape=(7, 7, 3)):
        super(nn_dueling, self).__init__()
        self.num_actions = num_actions
        self.optimizer = Adam(learning_rate=lr)
        self.loss_fn = MeanSquaredError()
        
        # Input and color-combination layer
        self.input_layer = InputLayer(input_shape=input_shape, name="input_layer")

        # Convolutional layers
        self.conv1 = Conv2D(32, (4, 4), strides=(1, 1), activation=None, padding='same', name="conv1_layer")
        self.bn1 = BatchNormalization(name="bn1_layer")
        self.dropout_conv1 = Dropout(0.3, name="dropout_conv1_layer")  # Dropout in the first convolutional layer

        self.conv2 = Conv2D(64, (3, 3), strides=(1, 1), activation=None, padding='same', name="conv2_layer")
        self.bn2 = BatchNormalization(name="bn2_layer")
        self.dropout_conv2 = Dropout(0.3, name="dropout_conv2_layer")  # Dropout in the second convolutional layer


        # Flatten
        self.flatten = Flatten(name="flatten_layer")
        self.dropout_flatten = Dropout(0.4, name="dropout_flatten_layer")  # Dropout after flattening

        # Shared dense layers
        self.dense_shared1 = Dense(64, activation='relu', name="shared_dense1", kernel_regularizer=l2(l2_regularization))
        self.dropout_shared = Dropout(0.4, name="dropout_shared_layer")  # Dropout in the shared layer

        # Value network (V)
        self.value_dense = Dense(32, activation='relu', name="value_dense", kernel_regularizer=l2(l2_regularization))
        self.dropout_value = Dropout(0.4, name="dropout_value_layer")  # Dropout in the value network
        self.value_output = Dense(1, activation='linear', name="value_output", kernel_regularizer=l2(l2_regularization))

        # Advantage network (A)
        self.advantage_dense = Dense(32, activation='relu', name="advantage_dense", kernel_regularizer=l2(l2_regularization))
        self.dropout_advantage = Dropout(0.4, name="dropout_advantage_layer")  # Dropout in the advantage network
        self.advantage_output = Dense(num_actions, activation='linear', name="advantage_output", kernel_regularizer=l2(l2_regularization))

    def call(self, inputs, training=False):
        x = self.input_layer(inputs)

        # Convolutional layer 1
        x = self.conv1(x)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x, name="relu1")
        x = self.dropout_conv1(x, training=training)  # Applied dropout

        # Convolutional layer 2
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x, name="relu2")
        x = self.dropout_conv2(x, training=training)  # Applied dropout
        

        x = self.flatten(x)
        x = self.dropout_flatten(x, training=training)  # Apply dropout after flattening

        # Shared layers
        x = self.dense_shared1(x)
        x = self.dropout_shared(x, training=training)  # Apply dropout in the shared layer

        # Value (V)
        v = self.value_dense(x)
        v = self.dropout_value(v, training=training)  # Apply dropout in the value network
        v = self.value_output(v)

        # Advantage (A)
        a = self.advantage_dense(x)
        a = self.dropout_advantage(a, training=training)  # Apply dropout in the advantage network
        a = self.advantage_output(a)

        # Combine V and A to calculate Q
        q = v + (a - tf.reduce_mean(a, axis=1, keepdims=True))
        return q
    
    def training_step(self, batch_data):
        states, actions, targetQ = batch_data
        with tf.GradientTape() as tape:
            Q_values = self(states, training=True)
            actions_onehot = tf.one_hot(actions, self.num_actions, dtype=tf.float32)
            Q = tf.reduce_sum(Q_values * actions_onehot, axis=1)
            loss = self.loss_fn(targetQ, Q)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return loss

    def predict_action(self, state):
        Q_values = self(state)
        return tf.argmax(Q_values, axis=1)[0].numpy()  # Return the action with the highest Q-value as a Python number

    def save_model(self, file_path):
        self.save(file_path)
        print(f"Modelo salvo em: {file_path}")

    def load_model(self, file_path):
        model_loaded = tf.keras.models.load_model(file_path)
        return model_loaded
    



In [26]:
# Model: nn_double
# ==========================================

class nn_double(tf.keras.Model):
    def __init__(self, num_actions=9, input_shape=(7, 7, 3), discount_factor=0.99):
        super(nn_double, self).__init__()
        self.num_actions = num_actions
        self.discount_factor = discount_factor

        # Optimizer and loss function
        self.optimizer = Adam(learning_rate=lr)
        self.loss_fn = MeanSquaredError()

        # Input layer and initial color processing
        self.input_layer = InputLayer(input_shape=input_shape, name="input_layer")


        # Convolutional layers of the main network
        self.conv1 = Conv2D(32, (4, 4), strides=(1, 1), activation=None, padding="same")
        self.batch_norm1 = BatchNormalization()
        self.relu1 = tf.keras.layers.ReLU()

        self.conv2 = Conv2D(64, (3, 3), strides=(1, 1), activation=None, padding="same")
        self.batch_norm2 = BatchNormalization()
        self.relu2 = tf.keras.layers.ReLU()


        # Dense layers
        self.flatten = Flatten()
        self.dense1 = Dense(64, activation="relu", kernel_regularizer=l2(l2_regularization))
        self.dense2 = Dense(32, activation="relu", kernel_regularizer=l2(l2_regularization))
        self.output_layer = Dense(num_actions, activation="linear", kernel_regularizer=l2(l2_regularization))

        # Target-network layers with the same modifications
        self.target_conv1 = Conv2D(32, (4, 4), strides=(1, 1), activation=None, padding="same")
        self.target_batch_norm1 = BatchNormalization()
        self.target_relu1 = tf.keras.layers.ReLU()

        self.target_conv2 = Conv2D(64, (3, 3), strides=(1, 1), activation=None, padding="same")
        self.target_batch_norm2 = BatchNormalization()
        self.target_relu2 = tf.keras.layers.ReLU()



        self.target_flatten = Flatten()
        self.target_dense1 = Dense(64, activation="relu", kernel_regularizer=l2(l2_regularization))
        self.target_dense2 = Dense(32, activation="relu", kernel_regularizer=l2(l2_regularization))
        self.target_output_layer = Dense(num_actions, activation="linear", kernel_regularizer=l2(l2_regularization))

    def call(self, inputs, training=False):
        """Chama a rede principal para inferência."""
        x = self.input_layer(inputs)

        x = self.conv1(x)
        x = self.batch_norm1(x, training=training)
        x = self.relu1(x)

        x = self.conv2(x)
        x = self.batch_norm2(x, training=training)
        x = self.relu2(x)


        x = self.flatten(x)
        x = self.dense1(x)
        x = self.dense2(x)
        return self.output_layer(x)

    def target_call(self, inputs, training=False):
        """Chama a rede-alvo para inferência."""
        x = self.input_layer(inputs)
        

        x = self.target_conv1(x)
        x = self.target_batch_norm1(x, training=training)
        x = self.target_relu1(x)

        x = self.target_conv2(x)
        x = self.target_batch_norm2(x, training=training)
        x = self.target_relu2(x)

        
        x = self.target_flatten(x)
        x = self.target_dense1(x)

        x = self.target_dense2(x)
        return self.target_output_layer(x)

    def update_target_network(self):
        """Sincroniza os pesos da rede principal para a rede-alvo."""
        self.target_conv1.set_weights(self.conv1.get_weights())
        self.target_batch_norm1.set_weights(self.batch_norm1.get_weights())
        self.target_conv2.set_weights(self.conv2.get_weights())
        self.target_batch_norm2.set_weights(self.batch_norm2.get_weights())
        self.target_dense1.set_weights(self.dense1.get_weights())
        self.target_dense2.set_weights(self.dense2.get_weights())
        self.target_output_layer.set_weights(self.output_layer.get_weights())

    def training_step(self, batch_data):
        """Realiza uma etapa de treinamento com Double DQN."""
        states, actions, targetQ = batch_data
        next_states, rewards, dones = targetQ
        with tf.GradientTape() as tape:
            # Main-network predictions
            Q_values = self(states, training=True)
            actions_onehot = tf.one_hot(actions, self.num_actions, dtype=tf.float32)
            Q = tf.reduce_sum(Q_values * actions_onehot, axis=1)

            # Double DQN: calculate the target value
            main_Q_values_next = self(next_states)  # Main network selects the best action
            next_actions = tf.argmax(main_Q_values_next, axis=1)
            target_Q_values_next = self.target_call(next_states)  # Target network calculates Q
            target_Q = tf.reduce_sum(target_Q_values_next * tf.one_hot(next_actions, self.num_actions), axis=1)
            target_Q = rewards + (1 - dones) * self.discount_factor * target_Q

            # Calculate the loss
            loss = self.loss_fn(target_Q, Q)

        # Compute gradients and update weights
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return loss

    def predict_action(self, state):
        """Prediz a melhor ação para um dado estado."""
        Q_values = self(tf.expand_dims(state, axis=0))
        return tf.argmax(Q_values, axis=1).numpy()[0]

    def save_model(self, file_path):
        """Salva os pesos do modelo principal."""
        self.save_weights(file_path)
        print(f"Modelo salvo em: {file_path}")

    def load_model(self, file_path):
        """Carrega os pesos do modelo principal."""
        self.load_weights(file_path)
        print(f"Modelo carregado de: {file_path}")






In [27]:
# Model: nn_radar_dqn
# ==========================================

class nn_radar_dqn(tf.keras.Model):
    def __init__(self, num_actions=9, input_shape=(7, 7, 3)):
        super(nn_radar_dqn, self).__init__()
        self.num_actions = num_actions
        self.optimizer = Adam(learning_rate=lr)
        self.loss_fn = MeanSquaredError()

        # Input and convolutional layers
        self.input_layer = InputLayer(input_shape=input_shape)
        self.color_comb_layer = ColorCombDepthwiseConv2D(kernel_size=(7, 7), activation='relu', name="color_comb_layer")

        # Convolutional layers
        self.conv1 = Conv2D(32, (4, 4), strides=(1, 1), activation=None, padding='same', name="conv1_layer")
        self.bn1 = BatchNormalization(name="bn1_layer")
        self.dropout_conv1 = Dropout(0.3, name="dropout_conv1_layer")  # Dropout after conv1

        self.conv2 = Conv2D(64, (3, 3), strides=(1, 1), activation=None, padding='same', name="conv2_layer")
        self.bn2 = BatchNormalization(name="bn2_layer")
        self.dropout_conv2 = Dropout(0.3, name="dropout_conv2_layer")  # Dropout after conv2

        # Spatial-attention layer
        self.spatial_attention = SpatialAttentionModule(kernel_size=4)

        # Flattening and dense layers
        self.flatten = Flatten(name="flatten_layer")
        self.dropout_flatten = Dropout(0.4, name="dropout_flatten_layer")  # Dropout after flattening

        self.dense1 = Dense(64, activation='relu', name="dense1_layer", kernel_regularizer=l2(l2_regularization))
        self.dropout_dense1 = Dropout(0.5, name="dropout_dense1_layer")  # Dropout after dense1
        
        self.dense2 = Dense(32, activation='relu', name="dense2_layer", kernel_regularizer=l2(l2_regularization))
        self.dropout_dense2 = Dropout(0.5, name="dropout_dense2_layer")  # Dropout after dense2

        self.dense_output = Dense(num_actions, activation='linear', name="dense_output_layer", kernel_regularizer=l2(l2_regularization))

    def call(self, inputs, training=False):
        x = self.input_layer(inputs)
        x = self.color_comb_layer(x)

        # Convolutional layer 1
        x = self.conv1(x)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x, name="relu1")
        x = self.dropout_conv1(x, training=training)  # Apply dropout after conv1

        # Convolutional layer 2
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x, name="relu2")
        x = self.dropout_conv2(x, training=training)  # Apply dropout after conv2

        # Spatial attention and flattening
        x = self.spatial_attention(x)
        x = self.flatten(x)
        x = self.dropout_flatten(x, training=training)  # Apply dropout after flattening

        # Dense layers
        x = self.dense1(x)
        x = self.dropout_dense1(x, training=training)  # Apply dropout after dense1
        x = self.dense2(x)
        x = self.dropout_dense2(x, training=training)  # Apply dropout after dense2

        Q_values = self.dense_output(x)
        return Q_values


    def training_step(self, batch_data):
        states, actions, targetQ = batch_data
        with tf.GradientTape() as tape:
            Q_values = self(states, training=True)
            actions_onehot = tf.one_hot(actions, self.num_actions, dtype=tf.float32)
            Q = tf.reduce_sum(Q_values * actions_onehot, axis=1)
            loss = self.loss_fn(targetQ, Q)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return loss

    def predict_action(self, state):
        Q_values = self(state)
        return tf.argmax(Q_values, axis=1)[0].numpy()  # Return the action with the highest Q-value as a Python number

    def save_model(self, file_path):
        self.save(file_path)
        print(f"Modelo salvo em: {file_path}")

    def load_model(self, file_path):
        model_loaded = tf.keras.models.load_model(file_path)
        return model_loaded
    

In [28]:
# Model: nn_radar_per
# ==========================================

class nn_radar_per(tf.keras.Model):
    def __init__(self, num_actions=9, input_shape=(7, 7, 3), discount_factor=0.99):
        super(nn_radar_per, self).__init__()
        self.num_actions = num_actions
        self.discount_factor = discount_factor
        self.optimizer = Adam(learning_rate=lr)
        self.loss_fn = MeanSquaredError()

        # Input and color-combination layer
        self.input_layer = InputLayer(input_shape=input_shape, name="input_layer")
        self.color_comb_layer = ColorCombDepthwiseConv2D(kernel_size=(7, 7), activation='relu', name="color_comb_layer")

        # Convolutional layers
        self.conv1 = Conv2D(32, (4, 4), strides=(1, 1), activation=None, padding='same', name="conv1_layer")
        self.bn1 = BatchNormalization(name="bn1_layer")
        self.dropout_conv1 = Dropout(0.3, name="dropout_conv1_layer")

        self.conv2 = Conv2D(64, (3, 3), strides=(1, 1), activation=None, padding='same', name="conv2_layer")
        self.bn2 = BatchNormalization(name="bn2_layer")
        self.dropout_conv2 = Dropout(0.3, name="dropout_conv2_layer")
        
        # Spatial-attention layer
        self.spatial_attention = SpatialAttentionModule(kernel_size=4)

        # Flattening and dense layers
        self.flatten = Flatten(name="flatten_layer")
        self.dropout_flatten = Dropout(0.3, name="dropout_flatten_layer")

        self.dense1 = Dense(64, activation='relu', name="dense1_layer", kernel_regularizer=l2(l2_regularization))
        self.dropout_dense1 = Dropout(0.4, name="dropout_dense1_layer")

        self.dense2 = Dense(32, activation='relu', name="dense2_layer", kernel_regularizer=l2(l2_regularization))
        self.dropout_dense2 = Dropout(0.4, name="dropout_dense2_layer")

        self.dense_output = Dense(num_actions, activation='linear', name="dense_output_layer", kernel_regularizer=l2(l2_regularization))

    def call(self, inputs, training=False):
        x = self.input_layer(inputs)
        x = self.color_comb_layer(x)

        x = self.conv1(x)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x, name="relu1")
        x = self.dropout_conv1(x, training=training)  # Apply dropout after conv1

        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x, name="relu2")
        x = self.dropout_conv2(x, training=training)  # Apply dropout after conv2

        x = self.spatial_attention(x)
        x = self.flatten(x)
        x = self.dropout_flatten(x, training=training)  # Apply dropout after flattening

        x = self.dense1(x)
        x = self.dropout_dense1(x, training=training)  # Apply dropout after dense1
        x = self.dense2(x)
        x = self.dropout_dense2(x, training=training)  # Apply dropout after dense2

        Q_values = self.dense_output(x)
        return Q_values
    

    def training_step(self, batch_data):
        states, actions, targetQ, weights = batch_data
        with tf.GradientTape() as tape:
            Q_values = self(states, training=True)
            actions_onehot = tf.one_hot(actions, self.num_actions, dtype=tf.float32)
            Q = tf.reduce_sum(Q_values * actions_onehot, axis=1)
            td_errors = targetQ - Q
            weighted_loss = tf.reduce_mean(weights * tf.square(td_errors))

        grads = tape.gradient(weighted_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        return weighted_loss  # Loss only


    def predict_action(self, state):
        Q_values = self(state)
        return tf.argmax(Q_values, axis=1)[0].numpy()  # Return the action with the highest Q-value as a Python number

    def save_model(self, file_path):
        self.save(file_path)
        print(f"Modelo salvo em: {file_path}")

    def load_model(self, file_path):
        model_loaded = tf.keras.models.load_model(file_path)
        return model_loaded




In [29]:
# Model: nn_radar_dueling
# ==========================================

class nn_radar_dueling(tf.keras.Model):
    def __init__(self, num_actions=9, input_shape=(7, 7, 3)):
        super(nn_radar_dueling, self).__init__()
        self.num_actions = num_actions
        self.optimizer = Adam(learning_rate=lr)
        self.loss_fn = MeanSquaredError()
        
        # Input and color-combination layer
        self.input_layer = InputLayer(input_shape=input_shape, name="input_layer")
        self.color_comb_layer = ColorCombDepthwiseConv2D(kernel_size=(7, 7), activation='relu', name="color_comb_layer")

        # Convolutional layers
        self.conv1 = Conv2D(32, (4, 4), strides=(1, 1), activation=None, padding='same', name="conv1_layer")
        self.bn1 = BatchNormalization(name="bn1_layer")
        self.dropout_conv1 = Dropout(0.3, name="dropout_conv1_layer")  # Dropout in the first convolutional layer

        self.conv2 = Conv2D(64, (3, 3), strides=(1, 1), activation=None, padding='same', name="conv2_layer")
        self.bn2 = BatchNormalization(name="bn2_layer")
        self.dropout_conv2 = Dropout(0.3, name="dropout_conv2_layer")  # Dropout in the second convolutional layer
     
        # Spatial-attention layer
        self.spatial_attention = SpatialAttentionModule(kernel_size=4)

        # Flatten
        self.flatten = Flatten(name="flatten_layer")
        self.dropout_flatten = Dropout(0.4, name="dropout_flatten_layer")  # Dropout after flattening

        # Shared dense layers
        self.dense_shared1 = Dense(64, activation='relu', name="shared_dense1", kernel_regularizer=l2(l2_regularization))
        self.dropout_shared = Dropout(0.4, name="dropout_shared_layer")  # Dropout in the shared layer

        # Value network (V)
        self.value_dense = Dense(32, activation='relu', name="value_dense", kernel_regularizer=l2(l2_regularization))
        self.dropout_value = Dropout(0.4, name="dropout_value_layer")  # Dropout in the value network
        self.value_output = Dense(1, activation='linear', name="value_output", kernel_regularizer=l2(l2_regularization))

        # Advantage network (A)
        self.advantage_dense = Dense(32, activation='relu', name="advantage_dense", kernel_regularizer=l2(l2_regularization))
        self.dropout_advantage = Dropout(0.4, name="dropout_advantage_layer")  # Dropout in the advantage network
        self.advantage_output = Dense(num_actions, activation='linear', name="advantage_output", kernel_regularizer=l2(l2_regularization))

    def call(self, inputs, training=False):
        x = self.input_layer(inputs)
        x = self.color_comb_layer(x)

        # Convolutional layer 1
        x = self.conv1(x)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x, name="relu1")
        x = self.dropout_conv1(x, training=training)  # Applied dropout

        # Convolutional layer 2
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x, name="relu2")
        x = self.dropout_conv2(x, training=training)  # Applied dropout
        
        # Spatial attention
        x = self.spatial_attention(x)
        x = self.flatten(x)
        x = self.dropout_flatten(x, training=training)  # Apply dropout after flattening

        # Shared layers
        x = self.dense_shared1(x)
        x = self.dropout_shared(x, training=training)  # Apply dropout in the shared layer

        # Value (V)
        v = self.value_dense(x)
        v = self.dropout_value(v, training=training)  # Apply dropout in the value network
        v = self.value_output(v)

        # Advantage (A)
        a = self.advantage_dense(x)
        a = self.dropout_advantage(a, training=training)  # Apply dropout in the advantage network
        a = self.advantage_output(a)

        # Combine V and A to calculate Q
        q = v + (a - tf.reduce_mean(a, axis=1, keepdims=True))
        return q
    
    def training_step(self, batch_data):
        states, actions, targetQ = batch_data
        with tf.GradientTape() as tape:
            Q_values = self(states, training=True)
            actions_onehot = tf.one_hot(actions, self.num_actions, dtype=tf.float32)
            Q = tf.reduce_sum(Q_values * actions_onehot, axis=1)
            loss = self.loss_fn(targetQ, Q)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return loss

    def predict_action(self, state):
        Q_values = self(state)
        return tf.argmax(Q_values, axis=1)[0].numpy()  # Return the action with the highest Q-value as a Python number

    def save_model(self, file_path):
        self.save(file_path)
        print(f"Modelo salvo em: {file_path}")

    def load_model(self, file_path):
        model_loaded = tf.keras.models.load_model(file_path)
        return model_loaded
    

In [30]:
# Model: nn_radar_double
# ==========================================

class nn_radar_double(tf.keras.Model):
    def __init__(self, num_actions=9, input_shape=(7, 7, 3), discount_factor=0.99):
        super(nn_radar_double, self).__init__()
        self.num_actions = num_actions
        self.discount_factor = discount_factor

        # Optimizer and loss function
        self.optimizer = Adam(learning_rate=lr)
        self.loss_fn = MeanSquaredError()

        # Input layer and initial color processing
        self.input_layer = InputLayer(input_shape=input_shape, name="input_layer")
        self.color_comb_layer = ColorCombDepthwiseConv2D(kernel_size=(7, 7), activation='relu', name="color_comb_layer")

        # Convolutional layers of the main network
        self.conv1 = Conv2D(32, (4, 4), strides=(1, 1), activation=None, padding="same")
        self.batch_norm1 = BatchNormalization()
        self.relu1 = tf.keras.layers.ReLU()

        self.conv2 = Conv2D(64, (3, 3), strides=(1, 1), activation=None, padding="same")
        self.batch_norm2 = BatchNormalization()
        self.relu2 = tf.keras.layers.ReLU()

        # Spatial-attention module
        self.spatial_attention = SpatialAttentionModule(kernel_size=4)

        # Dense layers
        self.flatten = Flatten()
        self.dense1 = Dense(64, activation="relu", kernel_regularizer=l2(l2_regularization))
        self.dense2 = Dense(32, activation="relu", kernel_regularizer=l2(l2_regularization))
        self.output_layer = Dense(num_actions, activation="linear", kernel_regularizer=l2(l2_regularization))

        # Target-network layers with the same modifications
        self.target_color_comb_layer = ColorCombDepthwiseConv2D(kernel_size=(7, 7), activation='relu', name="target_color_comb_layer")
        self.target_conv1 = Conv2D(32, (4, 4), strides=(1, 1), activation=None, padding="same")
        self.target_batch_norm1 = BatchNormalization()
        self.target_relu1 = tf.keras.layers.ReLU()

        self.target_conv2 = Conv2D(64, (3, 3), strides=(1, 1), activation=None, padding="same")
        self.target_batch_norm2 = BatchNormalization()
        self.target_relu2 = tf.keras.layers.ReLU()

        self.target_spatial_attention = SpatialAttentionModule(kernel_size=4)

        self.target_flatten = Flatten()
        self.target_dense1 = Dense(64, activation="relu", kernel_regularizer=l2(l2_regularization))
        self.target_dense2 = Dense(32, activation="relu", kernel_regularizer=l2(l2_regularization))
        self.target_output_layer = Dense(num_actions, activation="linear", kernel_regularizer=l2(l2_regularization))

    def call(self, inputs, training=False):
        """Chama a rede principal para inferência."""
        x = self.input_layer(inputs)
        x = self.color_comb_layer(x)

        x = self.conv1(x)
        x = self.batch_norm1(x, training=training)
        x = self.relu1(x)

        x = self.conv2(x)
        x = self.batch_norm2(x, training=training)
        x = self.relu2(x)

        x = self.spatial_attention(x)
        x = self.flatten(x)
        x = self.dense1(x)
        x = self.dense2(x)
        return self.output_layer(x)

    def target_call(self, inputs, training=False):
        """Chama a rede-alvo para inferência."""
        x = self.input_layer(inputs)
        x = self.target_color_comb_layer(x)

        x = self.target_conv1(x)
        x = self.target_batch_norm1(x, training=training)
        x = self.target_relu1(x)

        x = self.target_conv2(x)
        x = self.target_batch_norm2(x, training=training)
        x = self.target_relu2(x)

        x = self.target_spatial_attention(x)
        x = self.target_flatten(x)
        x = self.target_dense1(x)

        x = self.target_dense2(x)
        return self.target_output_layer(x)

    def update_target_network(self):
        """Sincroniza os pesos da rede principal para a rede-alvo."""
        self.target_color_comb_layer.set_weights(self.color_comb_layer.get_weights())
        self.target_conv1.set_weights(self.conv1.get_weights())
        self.target_batch_norm1.set_weights(self.batch_norm1.get_weights())
        self.target_conv2.set_weights(self.conv2.get_weights())
        self.target_batch_norm2.set_weights(self.batch_norm2.get_weights())
        self.target_dense1.set_weights(self.dense1.get_weights())
        self.target_dense2.set_weights(self.dense2.get_weights())
        self.target_output_layer.set_weights(self.output_layer.get_weights())

    def training_step(self, batch_data):
        """Realiza uma etapa de treinamento com Double DQN."""
        states, actions, targetQ = batch_data
        next_states, rewards, dones = targetQ
        with tf.GradientTape() as tape:
            # Main-network predictions
            Q_values = self(states, training=True)
            actions_onehot = tf.one_hot(actions, self.num_actions, dtype=tf.float32)
            Q = tf.reduce_sum(Q_values * actions_onehot, axis=1)

            # Double DQN: calculate the target value
            main_Q_values_next = self(next_states)  # Main network selects the best action
            next_actions = tf.argmax(main_Q_values_next, axis=1)
            target_Q_values_next = self.target_call(next_states)  # Target network calculates Q
            target_Q = tf.reduce_sum(target_Q_values_next * tf.one_hot(next_actions, self.num_actions), axis=1)
            target_Q = rewards + (1 - dones) * self.discount_factor * target_Q

            # Calculate the loss
            loss = self.loss_fn(target_Q, Q)

        # Compute gradients and update weights
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return loss

    def predict_action(self, state):
        """Prediz a melhor ação para um dado estado."""
        Q_values = self(tf.expand_dims(state, axis=0))
        return tf.argmax(Q_values, axis=1).numpy()[0]

    def save_model(self, file_path):
        """Salva os pesos do modelo principal."""
        self.save_weights(file_path)
        print(f"Modelo salvo em: {file_path}")

    def load_model(self, file_path):
        """Carrega os pesos do modelo principal."""
        self.load_weights(file_path)
        print(f"Modelo carregado de: {file_path}")





### SIMULATION
--------------

In [14]:
# Support
# ==========================================

def save_sim(task_name, episode, step, tot_predator, tot_prey, reward_total, done_total, output_dir="output_logs"):
    """
    Salva informações de treinamento em arquivos separados para cada tipo de agente (prey ou predator).
    """
    # Check and create the directory if necessary
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Iterate over agent types ('predator' and 'prey')
    for agent_type in ['predator', 'prey']:
        # Set the file name based on the agent type
        file_name = os.path.join(output_dir, f"{task_name}_{agent_type}_simlog.txt")

        # Build the text line to be written
        line = (
            f"Episode: {episode}, Step: {step}, "
            f"Population (Prey/Predator): {tot_prey}/{tot_predator}, "
            f"Total Reward ({agent_type}): {reward_total[agent_type]:.2f}, "
            f"Total Dones ({agent_type}): {done_total[agent_type]}\n"
        )

        # Open the file in append mode and write the line
        with open(file_name, "a") as file:
            file.write(line)

def normalize_channel(channel):

    if isinstance(channel, (list, tuple)):
        return tuple(int(value * 255) for value in channel)
    raise TypeError("O canal deve ser uma lista ou tupla de valores.")

def footprints(obstacles, agents, task_name, episode, step, grid_width, grid_height, save_obstacles=False):
    """
    Atualizado para incluir grid_width e grid_height em cada linha.
    Remove o arquivo existente se necessário.
    """
    model_dir = os.path.join(".", "final-models", "monitor")
    os.makedirs(model_dir, exist_ok=True)
    file_name = os.path.join(model_dir, f"sim_{task_name}_footprints.csv")
    header = "Episode,Step,Type,Is_Alive,R,G,B,X,Y,Grid_Width,Grid_Height\n"

    # Remove the existing file if necessary
    if os.path.exists(file_name) and step == 0 and episode == 0:
        os.remove(file_name)

    # Load existing data and initialize agent indices
    agent_indices = {}
    existing_data = set()

    if os.path.exists(file_name):
        with open(file_name, "r") as file:
            lines = file.readlines()
            if len(lines) > 1:
                for line in lines[1:]:
                    existing_data.add(line.strip())
                    index, _, _, agent_name, *_ = line.split(",")
                    agent_indices[agent_name] = int(index)

    # Set the file opening mode
    file_mode = "w" if not os.path.exists(file_name) else "a"
    index = max(agent_indices.values(), default=0) + 1

    with open(file_name, file_mode) as file:
        if file_mode == "w":
            file.write(header)

        # Write obstacle data
        if save_obstacles and step == 0:
            for obstacle in obstacles:
                normalized_channel = normalize_channel(obstacle.channel)
                line = (
                    f"{episode},{step},Obstacle,True,"
                    f"{normalized_channel[0]},{normalized_channel[1]},{normalized_channel[2]},"
                    f"{obstacle.x},{obstacle.y},{grid_width},{grid_height}\n"
                )
                if line.strip() not in existing_data:
                    file.write(line)
                    index += 1

        # Write agent data
        for agent in agents:
            normalized_channel = normalize_channel(agent.channel)
            is_alive = "True" if agent.is_alive else "False"
            line = (
                f"{episode},{step},{agent.name},{is_alive},"
                f"{normalized_channel[0]},{normalized_channel[1]},{normalized_channel[2]},"
                f"{agent.x},{agent.y},{grid_width},{grid_height}\n"
            )
            if line.strip() not in existing_data:
                file.write(line)
                agent_indices[agent.name] = index
                index += 1

def save_log_to_file(filename, episode, step, agent, pop_prey, pop_predator, action, reward, done, nearby, feedback):
    # Format the string as in the original print statement
    log_entry = (f"{episode}/{step}: {agent.name}, Pop: {pop_prey}/{pop_predator}, "
                 f"Breed:{agent.breed}, Channel: {agent.channel} Position: ({agent.x}, {agent.y}), Action: {action}, "
                 f"Reward: {reward}, Done: {done}, Nearby: {nearby}, Feedback: {feedback}")
    
    # Open the file in append mode to preserve previous logs
    with open(filename, 'a') as file:
        file.write(log_entry + '\n')  # Add a new line after each entry

def get_nearby_agents_count(agent, agents_list, max_distance=3):
    """
    Conta quantos agentes estão próximos a um agente específico, dentro de um raio especificado.

    Args:
        agent (object): O agente para o qual calcular a proximidade.
        agents_list (list): Lista de agentes no ambiente.
        max_distance (int): Distância máxima para considerar como "próximo". Default é 3.

    Returns:
        int: Número de agentes dentro da distância especificada do agente fornecido.
    """
    count = 0
    
    for other_agent in agents_list:
        # Ignore the current agent and agents that are not alive
        if other_agent == agent or agent.breed != other_agent.breed or not other_agent.is_alive:
            continue
        
        # Calculate the Chebyshev distance
        distance = max(abs(agent.x - other_agent.x), abs(agent.y - other_agent.y))
        
        if distance <= max_distance:
            count += 1
    
    return count



In [15]:
# SIMULATION
# ==========================================

class Sim:
    def __init__(self, task_name, category, env, num_predators, num_preys, limit_pop_predator, limit_pop_prey, model_nn_predator, model_nn_prey, num_episodes=100, num_steps=10):
        self.directory = f"C:/Users/beLIVE/IA/DECISYS"
        self.model_dir = os.path.join(".", "final-models", "sim")
        os.makedirs(self.model_dir, exist_ok=True)
        self.category = category
        self.task_name = task_name
        self.env = env
        self.num_predators = num_predators
        self.num_preys = num_preys
        self.num_episodes = num_episodes
        self.num_steps = num_steps
        self.model_nn_predator = model_nn_predator
        self.model_nn_prey = model_nn_prey
        self.count_episodes = 0
        self.limit_pop_predator = limit_pop_predator
        self.limit_pop_prey = limit_pop_prey

    def run(self):
        for episode in range(self.num_episodes):
            self.env.reset()

            episode_rewards = {'predator': 0, 'prey': 0}
            episode_dones = {'predator': 0, 'prey': 0}

            # Set a flag to save obstacles at the start of the episode
            save_obstacles_flag = True

            for step in range(self.num_steps):
                if not any(a.breed == 2 and a.is_alive for a in self.env.agents) or not any(a.breed == 0 and a.is_alive for a in self.env.agents):
                    break

                for agent in self.env.agents:
                    if not agent.is_alive:
                        continue

                    # Render state and predict action
                    state = self.env.render_agent(agent)
                    state_expanded = np.expand_dims(state, 0)
                    model = self.model_nn_prey if agent.breed == 2 else self.model_nn_predator
                    Q_values = model.predict(state_expanded, verbose=0)

                    
                    if self.category == "prqipydb":
                       action = np.argmax(Q_values[0]) if agent.breed == 0 else np.random.randint(0, 9)
                    elif self.category == "prdbpyqi":
                        action = np.argmax(Q_values[0]) if agent.breed == 2 else np.random.randint(0, 9)
                    else: 
                        action = np.argmax(Q_values[0])
                        

                    # Perform action
                    _, reward, done, feedback = agent.step(action)


                    # Update reward and terminal-state data
                    if agent.breed == 0:  # Predator
                        episode_rewards['predator'] += reward
                        episode_dones['predator'] += 1 if done else 0
                    elif agent.breed == 2:  # Prey
                        episode_rewards['prey'] += reward
                        episode_dones['prey'] += 1 if done else 0

                    agents_list = self.env.agents
                    nearby = get_nearby_agents_count(agent, agents_list)
                    pop_prey, pop_predator, _ = self.env.population_count()
                    # Print agent details
                    print(f"{episode}/{step}: {agent.name}, Pop: {pop_prey}/{pop_predator}, "
                          f"Breed:{agent.breed}, Color: {agent.channel}  Position: ({agent.x}, {agent.y}), Action: {action}, Reward: {reward}, Done: {done}, Nearby: {nearby}, Feedback: {feedback}")
                    save_log_to_file(self.task_name, episode, step, agent, pop_prey, pop_predator, action, reward, done, nearby, feedback)

                # Save agent footprints at each step
                footprints(self.env.obstacles, self.env.agents, self.task_name, episode, step, self.env.sizeX, self.env.sizeY,save_obstacles=save_obstacles_flag)

                # Disable the flag after saving obstacles for the first time
                if save_obstacles_flag: save_obstacles_flag = False
                
            save_sim(self.task_name, episode, step, pop_predator, pop_prey, episode_rewards, episode_dones, output_dir=self.model_dir)

            



### RUN
________________________

In [152]:
# SUPPORT - Extensive and Sparse

size = 20
num_preys = 15
num_predators = 10
num_episodes = 100
perc_obs = 0.05
num_steps = 10
v = 10

In [ ]:
# 1. Load - nn_dqn
#==========================

# Input shape for the models
input_shape = (7, 7, 3)
nn_model = "nn_dqn"

# Directory and paths for saved files
directory = "C:/Users/beLIVE/IA/DECISYS/final-models/last/model"
file_path_model_prey = os.path.join(directory, f'model_{nn_model}_prey.h5')
                                               
file_path_model_predator = os.path.join(directory, f'model_{nn_model}_predator.h5')

# ===========================
# Prey-model configuration
# ===========================
# Create the prey model
model_nn_prey = globals()[nn_model](num_actions=9, input_shape=input_shape)

# Initialize the prey model (main and target networks)
dummy_input = tf.random.uniform((1, *input_shape))
model_nn_prey(dummy_input)  # Activate the main network
#model_nn_prey.target_call(dummy_input)  # Activate the target network
#model_nn_prey.update_target_network()  # Create and synchronize the target network

# Load weights for the prey model
try:
    model_nn_prey.load_weights(file_path_model_prey)
    print("Pesos do modelo de presas carregados com sucesso!")
except Exception as e:
    print(f"Erro ao carregar os pesos do modelo de presas: {e}")

# ===========================
# Predator-model configuration
# ===========================
# Create the predator model
model_nn_predator = globals()[nn_model](num_actions=9, input_shape=input_shape)

# Initialize the predator model (main and target networks)
model_nn_predator(dummy_input)  # Activate the main network
#model_nn_predator.target_call(dummy_input)  # Activate the target network
#model_nn_predator.update_target_network()  # Create and synchronize the target network

# Load weights for the predator model
try:
    model_nn_predator.load_weights(file_path_model_predator)
    print("Pesos do modelo de predadores carregados com sucesso!")
except Exception as e:
    print(f"Erro ao carregar os pesos do modelo de predadores: {e}")

# ===========================
# Finalization
# ===========================
print("Modelos prontos para simulação!")


In [ ]:
# 2. RUN - prqipyqi
# ==========================================

category = "prqipyqi"

# Load model weights here if necessary
task_name = f"sim_{nn_model}{v}_sz{size}_s{num_steps}_py{num_preys}_pd{num_predators}_o{perc_obs}_{category}"


env = Env(sizeX=size, sizeY=size, ray=3)
num_obstacle = int(env.sizeX * env.sizeX * perc_obs)

# Populate the environment with obstacles
for o in range(num_obstacle):
    pos = env.new_position()
    if pos is not None:
        x, y = pos
        obstacle = Obstacle(x, y)
        env.add_obstacle(obstacle)
    else:
        print("Não foi possível encontrar uma posição nova para o obstacle.")

# Populate the environment with prey agents
for j in range(num_preys):
    pos = env.new_position()
    if pos is not None:
        x, y = pos
        prey = Prey(x, y, env, j, model_nn_prey)
        env.add_agent(prey)
    else:
        print("Não foi possível encontrar uma posição nova para o prey.")

# Populate the environment with predator agents
for i in range(num_predators):
    pos = env.new_position()
    if pos is not None:
        x, y = pos
        predator = Predator(x, y, env, i, model_nn_predator)
        env.add_agent(predator)
    else:
        print("Não foi possível encontrar uma posição nova para o predator.")
limit_pop_predator = 0
limit_pop_prey = 0

# Run the test session
sim = Sim(task_name, category, env, num_predators, num_preys, limit_pop_predator, limit_pop_prey, model_nn_predator, model_nn_prey, num_episodes, num_steps)

sim.run()


In [ ]:
# 3 - Load nn_per
#==========================

# Input shape for the models
input_shape = (7, 7, 3)
nn_model = "nn_per"


# Directory and paths for saved files
directory = "C:/Users/beLIVE/IA/DECISYS/final-models/last/model"
file_path_model_prey = os.path.join(directory, f'model_{nn_model}_prey.h5')
                                               
file_path_model_predator = os.path.join(directory, f'model_{nn_model}_predator.h5')

# ===========================
# Prey-model configuration
# ===========================
# Create the prey model
model_nn_prey = globals()[nn_model](num_actions=9, input_shape=input_shape)

# Initialize the prey model (main and target networks)
dummy_input = tf.random.uniform((1, *input_shape))
model_nn_prey(dummy_input)  # Activate the main network
#model_nn_prey.target_call(dummy_input)  # Activate the target network
#model_nn_prey.update_target_network()  # Create and synchronize the target network

# Load weights for the prey model
try:
    model_nn_prey.load_weights(file_path_model_prey)
    print("Pesos do modelo de presas carregados com sucesso!")
except Exception as e:
    print(f"Erro ao carregar os pesos do modelo de presas: {e}")

# ===========================
# Predator-model configuration
# ===========================
# Create the predator model
model_nn_predator = globals()[nn_model](num_actions=9, input_shape=input_shape)

# Initialize the predator model (main and target networks)
model_nn_predator(dummy_input)  # Activate the main network
#model_nn_predator.target_call(dummy_input)  # Activate the target network
#model_nn_predator.update_target_network()  # Create and synchronize the target network

# Load weights for the predator model
try:
    model_nn_predator.load_weights(file_path_model_predator)
    print("Pesos do modelo de predadores carregados com sucesso!")
except Exception as e:
    print(f"Erro ao carregar os pesos do modelo de predadores: {e}")

# ===========================
# Finalization
# ===========================
print("Modelos prontos para simulação!")


In [ ]:
# 4. RUN - prqipyqi
# ==========================================

category = "prqipyqi"


# Load model weights here if necessary
task_name = f"sim_{nn_model}{v}_sz{size}_s{num_steps}_py{num_preys}_pd{num_predators}_o{perc_obs}_{category}"

env = Env(sizeX=size, sizeY=size, ray=3)
num_obstacle = int(env.sizeX * env.sizeX * perc_obs)

# Populate the environment with obstacles
for o in range(num_obstacle):
    pos = env.new_position()
    if pos is not None:
        x, y = pos
        obstacle = Obstacle(x, y)
        env.add_obstacle(obstacle)
    else:
        print("Não foi possível encontrar uma posição nova para o obstacle.")

# Populate the environment with prey agents
for j in range(num_preys):
    pos = env.new_position()
    if pos is not None:
        x, y = pos
        prey = Prey(x, y, env, j, model_nn_prey)
        env.add_agent(prey)
    else:
        print("Não foi possível encontrar uma posição nova para o prey.")

# Populate the environment with predator agents
for i in range(num_predators):
    pos = env.new_position()
    if pos is not None:
        x, y = pos
        predator = Predator(x, y, env, i, model_nn_predator)
        env.add_agent(predator)
    else:
        print("Não foi possível encontrar uma posição nova para o predator.")
limit_pop_predator = 0
limit_pop_prey = 0

# Run the test session
sim = Sim(task_name, category, env, num_predators, num_preys, limit_pop_predator, limit_pop_prey, model_nn_predator, model_nn_prey, num_episodes, num_steps)

sim.run()


In [ ]:
# 5 - Load nn_dueling
#==========================

# Input shape for the models
input_shape = (7, 7, 3)
nn_model = "nn_dueling"


# Directory and paths for saved files
directory = "C:/Users/beLIVE/IA/DECISYS/final-models/last/model"
file_path_model_prey = os.path.join(directory, f'model_{nn_model}_prey.h5')
                                               
file_path_model_predator = os.path.join(directory, f'model_{nn_model}_predator.h5')

# ===========================
# Prey-model configuration
# ===========================
# Create the prey model
model_nn_prey = globals()[nn_model](num_actions=9, input_shape=input_shape)

# Initialize the prey model (main and target networks)
dummy_input = tf.random.uniform((1, *input_shape))
model_nn_prey(dummy_input)  # Activate the main network
#model_nn_prey.target_call(dummy_input)  # Activate the target network
#model_nn_prey.update_target_network()  # Create and synchronize the target network

# Load weights for the prey model
try:
    model_nn_prey.load_weights(file_path_model_prey)
    print("Pesos do modelo de presas carregados com sucesso!")
except Exception as e:
    print(f"Erro ao carregar os pesos do modelo de presas: {e}")

# ===========================
# Predator-model configuration
# ===========================
# Create the predator model
model_nn_predator = globals()[nn_model](num_actions=9, input_shape=input_shape)

# Initialize the predator model (main and target networks)
model_nn_predator(dummy_input)  # Activate the main network
#model_nn_predator.target_call(dummy_input)  # Activate the target network
#model_nn_predator.update_target_network()  # Create and synchronize the target network

# Load weights for the predator model
try:
    model_nn_predator.load_weights(file_path_model_predator)
    print("Pesos do modelo de predadores carregados com sucesso!")
except Exception as e:
    print(f"Erro ao carregar os pesos do modelo de predadores: {e}")

# ===========================
# Finalization
# ===========================
print("Modelos prontos para simulação!")


In [ ]:
# 6. RUN - prqipyqi
# ==========================================

category = "prqipyqi"

# Load model weights here if necessary
task_name = f"sim_{nn_model}{v}_sz{size}_s{num_steps}_py{num_preys}_pd{num_predators}_o{perc_obs}_{category}"


env = Env(sizeX=size, sizeY=size, ray=3)
num_obstacle = int(env.sizeX * env.sizeX * perc_obs)

# Populate the environment with obstacles
for o in range(num_obstacle):
    pos = env.new_position()
    if pos is not None:
        x, y = pos
        obstacle = Obstacle(x, y)
        env.add_obstacle(obstacle)
    else:
        print("Não foi possível encontrar uma posição nova para o obstacle.")

# Populate the environment with prey agents
for j in range(num_preys):
    pos = env.new_position()
    if pos is not None:
        x, y = pos
        prey = Prey(x, y, env, j, model_nn_prey)
        env.add_agent(prey)
    else:
        print("Não foi possível encontrar uma posição nova para o prey.")

# Populate the environment with predator agents
for i in range(num_predators):
    pos = env.new_position()
    if pos is not None:
        x, y = pos
        predator = Predator(x, y, env, i, model_nn_predator)
        env.add_agent(predator)
    else:
        print("Não foi possível encontrar uma posição nova para o predator.")
limit_pop_predator = 0
limit_pop_prey = 0

# Run the test session
sim = Sim(task_name, category, env, num_predators, num_preys, limit_pop_predator, limit_pop_prey, model_nn_predator, model_nn_prey, num_episodes, num_steps)

sim.run()


In [ ]:
# 7 - Load nn_double
#==========================

# Input shape for the models
input_shape = (7, 7, 3)
nn_model = "nn_double"


# Directory and paths for saved files
directory = "C:/Users/beLIVE/IA/DECISYS/final-models/last/model"
file_path_model_prey = os.path.join(directory, f'model_{nn_model}_prey.h5')
                                               
file_path_model_predator = os.path.join(directory, f'model_{nn_model}_predator.h5')

# ===========================
# Prey-model configuration
# ===========================
# Create the prey model
model_nn_prey = globals()[nn_model](num_actions=9, input_shape=input_shape)

# Initialize the prey model (main and target networks)
dummy_input = tf.random.uniform((1, *input_shape))
model_nn_prey(dummy_input)  # Activate the main network
model_nn_prey.target_call(dummy_input)  # Activate the target network
model_nn_prey.update_target_network()  # Create and synchronize the target network

# Load weights for the prey model
try:
    model_nn_prey.load_weights(file_path_model_prey)
    print("Pesos do modelo de presas carregados com sucesso!")
except Exception as e:
    print(f"Erro ao carregar os pesos do modelo de presas: {e}")

# ===========================
# Predator-model configuration
# ===========================
# Create the predator model
model_nn_predator = globals()[nn_model](num_actions=9, input_shape=input_shape)

# Initialize the predator model (main and target networks)
model_nn_predator(dummy_input)  # Activate the main network
model_nn_predator.target_call(dummy_input)  # Activate the target network
model_nn_predator.update_target_network()  # Create and synchronize the target network

# Load weights for the predator model
try:
    model_nn_predator.load_weights(file_path_model_predator)
    print("Pesos do modelo de predadores carregados com sucesso!")
except Exception as e:
    print(f"Erro ao carregar os pesos do modelo de predadores: {e}")

# ===========================
# Finalization
# ===========================
print("Modelos prontos para simulação!")


In [ ]:
# 8. RUN - prqipyqi
# ==========================================

category = "prqipyqi"

# Load model weights here if necessary
task_name = f"sim_{nn_model}{v}_sz{size}_s{num_steps}_py{num_preys}_pd{num_predators}_o{perc_obs}_{category}"


env = Env(sizeX=size, sizeY=size, ray=3)
num_obstacle = int(env.sizeX * env.sizeX * perc_obs)

# Populate the environment with obstacles
for o in range(num_obstacle):
    pos = env.new_position()
    if pos is not None:
        x, y = pos
        obstacle = Obstacle(x, y)
        env.add_obstacle(obstacle)
    else:
        print("Não foi possível encontrar uma posição nova para o obstacle.")

# Populate the environment with prey agents
for j in range(num_preys):
    pos = env.new_position()
    if pos is not None:
        x, y = pos
        prey = Prey(x, y, env, j, model_nn_prey)
        env.add_agent(prey)
    else:
        print("Não foi possível encontrar uma posição nova para o prey.")

# Populate the environment with predator agents
for i in range(num_predators):
    pos = env.new_position()
    if pos is not None:
        x, y = pos
        predator = Predator(x, y, env, i, model_nn_predator)
        env.add_agent(predator)
    else:
        print("Não foi possível encontrar uma posição nova para o predator.")
limit_pop_predator = 0
limit_pop_prey = 0

# Run the test session
sim = Sim(task_name, category, env, num_predators, num_preys, limit_pop_predator, limit_pop_prey, model_nn_predator, model_nn_prey, num_episodes, num_steps)

sim.run()
